# Tema: Tablas managed y external; Delta y Parquet

## Objetivos
Separar formato y propiedad del almacenamiento; observar DROP y registrar datos existentes.

## Conceptos importantes para el examen
Managed/external define gestión del ciclo de vida; Delta/Parquet define formato. DROP external elimina metadatos, conserva archivos; DROP managed inicia el ciclo de eliminación con ventana de recuperación.

**Dificultad:** Intermedio · **Tiempo estimado:** 75 min.

La parte external requiere una external location de prácticas con CREATE EXTERNAL TABLE y permisos de lectura/escritura de archivos. Deja external_base vacío si no tienes acceso: podrás completar la comparación managed y la simulación de archivos. Nunca registres una tabla external bajo /Volumes.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_09_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)
dbutils.widgets.text("external_base", "")
EXTERNAL_BASE = dbutils.widgets.get("external_base").rstrip("/")
if EXTERNAL_BASE:
    assert EXTERNAL_BASE.startswith(("s3://", "abfss://", "gs://")), "Usa una ruta de external location autorizada"
    assert "'" not in EXTERNAL_BASE and "`" not in EXTERNAL_BASE
    EXT = EXTERNAL_BASE + "/dea_" + RUN_ID

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Tabla managed

In [ ]:
%sql
CREATE TABLE managed_demo USING DELTA AS SELECT * FROM employees;
DESCRIBE EXTENDED managed_demo;

### 2. Comparación de formatos
UC no admite una tabla managed Parquet como equivalente de una managed Delta. Parquet se compara como external; sin permisos se compara solo el conjunto de archivos.

In [ ]:
employees.write.mode("overwrite").parquet(BASE + "/parquet_files")
employees.write.format("delta").mode("overwrite").save(BASE + "/delta_files")
display(dbutils.fs.ls(BASE + "/parquet_files"))
display(dbutils.fs.ls(BASE + "/delta_files"))
if EXTERNAL_BASE:
    employees.write.format("delta").mode("overwrite").save(EXT + "/delta")
    employees.write.mode("overwrite").parquet(EXT + "/parquet")
    spark.sql(f"CREATE TABLE external_delta USING DELTA LOCATION '{EXT}/delta'")
    spark.sql(f"CREATE TABLE external_parquet USING PARQUET LOCATION '{EXT}/parquet'")
    display(spark.sql("DESCRIBE EXTENDED external_delta"))
else:
    print("Comparación por archivos completa. Registro external pendiente de external_base.")

### 3. DROP managed
No presupongas eliminación física instantánea. Consulta Catalog Explorer; UC conserva una ventana de recuperación.

In [ ]:
%sql
DROP TABLE managed_demo;
SHOW TABLES;

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea managed_practice con USING DELTA y confirma MANAGED en DESCRIBE EXTENDED.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Si configuraste external_base, elimina external_delta y demuestra que siguen los archivos y sus 18 filas. Sin permisos, demuestra solo lectura Delta de la copia del volumen y registra la limitación.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Vuelve a registrar la tabla external sobre la misma ubicación; alternativamente vuelve a leer la copia del volumen.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Compara Parquet y Delta buscando _delta_log y leyendo ambos con su lector correcto.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Practica conversión external→managed si dispones de external_delta y DBR 17.3+ compatible; si no, crea una copia managed y explica que tiene identidad e historial propios.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** El nombre del formato no identifica propiedad.

**Pista 2:** DROP TABLE no borra los datos external.

**Pista 3:** No vuelvas a escribir los datos.

**Pista 4:** La semántica transaccional corresponde al lector Delta.

**Pista 5:** ALTER TABLE SET MANAGED requiere soporte y permisos adicionales; la copia CTAS es portable.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE managed_practice USING DELTA AS SELECT * FROM employees;
DESCRIBE EXTENDED managed_practice;

### Solución 2

In [ ]:
if EXTERNAL_BASE:
    spark.sql("DROP TABLE IF EXISTS external_delta")
    display(dbutils.fs.ls(EXT + "/delta"))
    assert spark.read.format("delta").load(EXT + "/delta").count() == 18
else:
    assert spark.read.format("delta").load(BASE + "/delta_files").count() == 18
    print("No se ha probado DROP external: requiere external location.")

### Solución 3

In [ ]:
if EXTERNAL_BASE:
    spark.sql(f"CREATE TABLE IF NOT EXISTS external_delta USING DELTA LOCATION '{EXT}/delta'")
    assert spark.table("external_delta").count() == 18
else:
    display(spark.read.format("delta").load(BASE + "/delta_files"))

### Solución 4

In [ ]:
assert spark.read.parquet(BASE + "/parquet_files").count() == 18
assert spark.read.format("delta").load(BASE + "/delta_files").count() == 18
assert any(x.name.rstrip("/") == "_delta_log" for x in dbutils.fs.ls(BASE + "/delta_files"))
# UPDATE/MERGE no son operaciones transaccionales sobre una tabla Parquet simple.

### Solución 5

In [ ]:
# Ruta portable: copia, no conversión in situ.
if EXTERNAL_BASE:
    spark.sql("CREATE OR REPLACE TABLE managed_from_external USING DELTA AS SELECT * FROM external_delta")
else:
    spark.read.format("delta").load(BASE + "/delta_files").write.format("delta").mode("overwrite").saveAsTable("managed_from_external")
# Ampliación opcional, tras revisar requisitos:
# spark.sql("ALTER TABLE external_delta SET MANAGED")
# La conversión inversa no es un ALTER genérico simétrico:
# exporta a una nueva ruta autorizada y registra otra tabla external.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué conserva DROP TABLE de una external?

A. Solo el nombre

B. Los archivos subyacentes

C. Todos los permisos

D. La entrada del metastore

### Pregunta 2
¿Qué combinación es válida en UC para esta práctica?

A. Managed Parquet obligatoria

B. Delta significa siempre managed

C. External Parquet

D. External bajo cualquier /Volumes

### Pregunta 3
¿Qué añade Delta sobre los archivos de datos?

A. Un registro transaccional para snapshots y ACID

B. Un reemplazo de Unity Catalog

C. Un warehouse automático

D. Un fichero CSV obligatorio

### Respuestas y explicación
**1. B** — El catálogo deja de registrar la tabla, pero no elimina sus datos.

**2. C** — Formato y tipo de gestión son dimensiones diferentes.

**3. A** — El log define los estados de la tabla.

### Documentación oficial
- [Managed](https://docs.databricks.com/aws/en/tables/managed)
- [External](https://docs.databricks.com/aws/en/tables/external)
- [Conversión](https://docs.databricks.com/aws/en/tables/convert-external-managed)

## PARTE 6 - RETO FINAL
Diseña un experimento para dar de baja el registro de una tabla y recuperarlo conservando datos. Documenta qué cambia para managed y external y qué operaciones realmente pudiste ejecutar.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
